In [ ]:
import os, time, math, json, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

# =========================================================================
# COLAB-OPTIMIZED CONFIGURATION
# =========================================================================
class Config:
    SPARSITY = 0.95        # Fixed at 95% per request
    SEEDS = [8,42,123]   # Standard seeds for statistical significance
    EPOCHS = 100
    BATCH = 128
    LR = 0.05
    RHO = 0.05
    WD = 5e-4

    # Pathing (Colab friendly)
    RESULTS_DIR = Path('/content/sparse_sam_results')
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

CFG = Config()
CFG.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================================
# MODEL: RESNET-18 (Optimized for CIFAR)
# =========================================================================
class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride, bias=False),
                nn.BatchNorm2d(planes))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNetCIFAR(nn.Module):
    def __init__(self):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._layer(64, 2, 1); self.layer2 = self._layer(128, 2, 2)
        self.layer3 = self._layer(256, 2, 2); self.layer4 = self._layer(512, 2, 2)
        self.fc = nn.Linear(512, 10)
    def _layer(self, planes, blocks, stride):
        layers = [BasicBlock(self.in_planes, planes, stride)]
        self.in_planes = planes
        for _ in range(1, blocks): layers.append(BasicBlock(self.in_planes, planes))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        return self.fc(F.avg_pool2d(x, 4).view(x.size(0), -1))

# =========================================================================
# OPTIMIZER: ISO-NORM SPARSE SAM
# =========================================================================
class SparseSAM(optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, sparsity=0.95, isonorm=True, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.sparsity = sparsity
        self.isonorm = isonorm

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        # 1. Calculate Grad Norm
        grad_norm = torch.norm(torch.stack([p.grad.norm(2) for g in self.param_groups for p in g['params'] if p.grad is not None]), 2)
        dense_scale = self.defaults['rho'] / (grad_norm + 1e-12)

        # 2. Sparse Perturbation with Optional Iso-Norm
        total_sq_norm = torch.tensor(0.0, device=CFG.DEVICE)
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                e_dense = p.grad * dense_scale
                k = max(1, int(e_dense.numel() * (1 - self.sparsity)))

                flat = e_dense.abs().view(-1)
                _, indices = torch.topk(flat, k)
                mask = torch.zeros_like(flat)
                mask[indices] = 1.0

                e_sparse = e_dense * mask.view_as(p)
                self.state[p]['e_w'] = e_sparse
                total_sq_norm += e_sparse.pow(2).sum()

        renorm = (self.defaults['rho'] / (torch.sqrt(total_sq_norm) + 1e-12)) if self.isonorm else 1.0

        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                self.state[p]['e_w'] *= renorm
                p.add_(self.state[p]['e_w'])
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                p.sub_(self.state[p]['e_w'])
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

    def zero_grad(self): self.base_optimizer.zero_grad()

# =========================================================================
# EXPERIMENT ENGINE
# =========================================================================
def run_experiment(method, seed, isonorm=False):
    fname = f"{method}_s{seed}_sp95.json"
    save_path = CFG.RESULTS_DIR / fname
    if save_path.exists():
        print(f"Skipping {fname} (Exists)"); return

    # Setup Seeds & Data
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tf = T.Compose([T.RandomCrop(32,4), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize((0.491, 0.482, 0.446), (0.202, 0.199, 0.201))])
    train_full = torchvision.datasets.CIFAR10('./data', True, tf, download=True)
    idx = []
    for c in range(10):
        c_idx = np.where(np.array(train_full.targets) == c)[0]
        idx.extend(c_idx[:len(c_idx)//2]) # 50% subset
    train_dl = DataLoader(Subset(train_full, idx), CFG.BATCH, shuffle=True, num_workers=2)
    test_dl = DataLoader(torchvision.datasets.CIFAR10('./data', False, T.Compose([T.ToTensor(), T.Normalize((0.491, 0.482, 0.446), (0.202, 0.199, 0.201))]), download=True), CFG.BATCH)

    model = ResNetCIFAR().to(CFG.DEVICE)
    opt = SparseSAM(model.parameters(), optim.SGD, rho=CFG.RHO, sparsity=CFG.SPARSITY, isonorm=isonorm, lr=CFG.LR, momentum=0.9, weight_decay=CFG.WD)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt.base_optimizer, CFG.EPOCHS)
    crit = nn.CrossEntropyLoss()

    history = {'train_acc': [], 'test_acc': [], 'perturb_norm': [], 'generalization_gap': []}

    print(f"\nTraining {fname}...")
    for epoch in range(CFG.EPOCHS):
        model.train(); corr, tot, p_norms = 0, 0, []
        for x, y in train_dl:
            x, y = x.to(CFG.DEVICE), y.to(CFG.DEVICE)
            out = model(x); loss = crit(out, y)
            loss.backward(); opt.first_step(True)

            # --- FIXED LOGIC HERE ---
            if epoch % 10 == 0 and len(p_norms) < 1:
                # Calculate norm properly by iterating params first
                sq_sum = sum(opt.state[p]['e_w'].pow(2).sum().item()
                           for g in opt.param_groups
                           for p in g['params']
                           if 'e_w' in opt.state[p])
                p_norms.append(math.sqrt(sq_sum))
            # ------------------------

            crit(model(x), y).backward(); opt.second_step(True)
            corr += (out.argmax(1) == y).sum().item(); tot += y.size(0)

        sched.step()
        model.eval(); t_corr, t_tot = 0, 0
        with torch.no_grad():
            for x, y in test_dl:
                x, y = x.to(CFG.DEVICE), y.to(CFG.DEVICE)
                t_corr += (model(x).argmax(1) == y).sum().item(); t_tot += y.size(0)

        # Save metrics
        train_acc = 100. * corr / tot
        test_acc = 100. * t_corr / t_tot
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        history['generalization_gap'].append(train_acc - test_acc)
        history['perturb_norm'].append(np.mean(p_norms) if p_norms else 0.0)

        if epoch % 25 == 0: print(f"Ep {epoch} | Train: {history['train_acc'][-1]:.1f}% | Test: {history['test_acc'][-1]:.1f}%")

    with open(save_path, 'w') as f: json.dump(history, f)
    print(f"Finished {fname}")

# =========================================================================
# INDEPENDENT RUNNERS
# =========================================================================
# In Colab, you can comment/uncomment these to run specifically
for s in CFG.SEEDS:
    run_experiment('sgd', s)
    # run_experiment('sam_dense', s)
    # run_experiment('ssam_standard', s, isonorm=False) # Without norm correction
    # run_experiment('ssam_isonorm', s, isonorm=True)   # With our proposed correction

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from scipy import stats

# =========================================================================
# CONFIGURATION
# =========================================================================
RESULTS_DIR = Path('/content/')
SEEDS = [8, 42, 123]

# Updated to include ALL methods
METHODS = {
    'sgd':             {'label': 'SGD',           'color': 'black',     'marker': 'x', 'ls': ':'},
    'sam_dense':       {'label': 'Dense SAM',     'color': 'blue',      'marker': 'd', 'ls': '--'},
    'ssam_standard':   {'label': 'Standard SSAM', 'color': '#E74C3C',   'marker': 'o', 'ls': '-'},
    'ssam_isonorm':    {'label': 'Iso-Norm SSAM', 'color': '#2ECC71',   'marker': 's', 'ls': '-'}
}

# Expected theoretical values
RHO = 0.05
SPARSITY = 0.95
THEORETICAL_STANDARD_NORM = RHO * np.sqrt(1 - SPARSITY)

# =========================================================================
# DATA LOADING & AGGREGATION
# =========================================================================
def load_all_runs():
    """Load all experimental runs and compute statistics"""
    data = {}
    print(f"Loading data from {RESULTS_DIR}...")

    for method in METHODS.keys():
        # Initialize lists for all metrics
        data[method] = {
            'test_acc': [], 'train_acc': [],
            'train_loss': [], 'test_loss': [],
            'perturb_norm': []
        }

        for seed in SEEDS:
            filepath = RESULTS_DIR / f"{method}_s{seed}_sp95.json"

            if not filepath.exists():
                # Try fallback for typos (missing dot)
                filepath = RESULTS_DIR / f"{method}_s{seed}_sp95json"
                if not filepath.exists():
                    print(f"⚠️  Missing: {method}_s{seed}")
                    continue

            with open(filepath) as f:
                run = json.load(f)
                # Load all keys safely
                for key in data[method].keys():
                    val = run.get(key, [])
                    data[method][key].append(val)

    # Compute statistics (mean, std)
    stats_data = {}
    for method, runs in data.items():
        if not runs['test_acc']: continue

        stats_data[method] = {}
        # Iterate over all metrics including losses
        for metric in ['test_acc', 'train_acc', 'train_loss', 'test_loss', 'perturb_norm']:
            if not runs[metric] or not runs[metric][0]:
                stats_data[method][metric] = {'mean': [], 'std': [], 'runs': []}
                continue

            # Pad arrays
            min_len = min(len(r) for r in runs[metric])
            truncated_runs = [r[:min_len] for r in runs[metric]]

            arr = np.array(truncated_runs)
            stats_data[method][metric] = {
                'mean': np.mean(arr, axis=0),
                'std': np.std(arr, axis=0),
                'runs': arr
            }

    return data, stats_data

# =========================================================================
# PLOTTING FUNCTIONS
# =========================================================================

def plot_test_accuracy(stats_data, save_path='fig1_test_accuracy_all.png'):
    fig, ax = plt.subplots(figsize=(10, 6))
    for method, style in METHODS.items():
        if method not in stats_data: continue
        d = stats_data[method]['test_acc']
        if len(d['mean']) == 0: continue

        epochs = np.arange(len(d['mean'])) + 1
        ax.plot(epochs, d['mean'], label=style['label'], color=style['color'], ls=style['ls'], lw=2)
        ax.fill_between(epochs, d['mean'] - d['std'], d['mean'] + d['std'], alpha=0.1, color=style['color'])

    ax.set_ylabel('Test Accuracy (%)'); ax.set_xlabel('Epoch')
    ax.set_title('Test Accuracy Comparison', fontweight='bold')
    ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(RESULTS_DIR / save_path, dpi=300); plt.show()

def plot_loss_curves(stats_data, save_path='fig4_loss_curves_all.png'):
    """Plots Train and Test Loss side-by-side"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Plot Training Loss
    for method, style in METHODS.items():
        if method not in stats_data: continue
        d = stats_data[method]['train_loss']
        if len(d['mean']) == 0: continue

        epochs = np.arange(len(d['mean'])) + 1
        ax1.plot(epochs, d['mean'], label=style['label'], color=style['color'], ls=style['ls'], lw=2)
        # Log scale often helps visualize loss convergence better
        ax1.set_yscale('log')

    ax1.set_title('Training Loss (Log Scale)', fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross Entropy Loss')
    ax1.grid(True, alpha=0.3, which="both")

    # Plot Test Loss
    for method, style in METHODS.items():
        if method not in stats_data: continue
        d = stats_data[method]['test_loss']
        if len(d['mean']) == 0: continue

        epochs = np.arange(len(d['mean'])) + 1
        ax2.plot(epochs, d['mean'], label=style['label'], color=style['color'], ls=style['ls'], lw=2)

    ax2.set_title('Test Loss', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout(); plt.savefig(RESULTS_DIR / save_path, dpi=300)
    print(f"✅ Saved: {save_path}")
    plt.show()

# def plot_perturbation_norms(stats_data, save_path='fig2_perturb_norms_all.png'):
#     fig, ax = plt.subplots(figsize=(10, 6))
#     ax.axhline(y=RHO, color='black', ls='--', label=f'Target ρ={RHO}')

#     for method, style in METHODS.items():
#         if method == 'sgd' or method not in stats_data: continue
#         d = stats_data[method]['perturb_norm']
#         if len(d['mean']) == 0: continue

#         # Handle sparse logging (e.g. every 10 epochs)
#         x = np.linspace(0, 100, len(d['mean']))
#         ax.plot(x, d['mean'], label=style['label'], color=style['color'], ls=style['ls'], lw=2, marker=style['marker'])

#     ax.set_title('Perturbation Norm Stability', fontweight='bold')
#     ax.set_ylabel('Norm ||ε||₂'); ax.set_xlabel('Epoch')
#     ax.legend(loc='right'); ax.grid(True, alpha=0.3); ax.set_ylim(0, 0.06)
#     plt.tight_layout(); plt.savefig(RESULTS_DIR / save_path, dpi=300); plt.show()
def plot_perturbation_norms(stats_data, save_path='fig2_perturb_norms_clean.png'):
    fig, ax = plt.subplots(figsize=(10, 6))

    # 1. Plot Target Lines
    ax.axhline(y=RHO, color='black', linestyle='--', linewidth=1.5,
               label=f'Target ρ = {RHO}', zorder=1)

    # 2. Plot Each Method
    for method, style in METHODS.items():
        if method not in stats_data: continue
        if method == 'sgd': continue # SGD has no perturbation norm

        norms = stats_data[method]['perturb_norm']['mean']

        # --- THE FIX: Filter out zeros ---
        # We create a list of (epoch, value) tuples ONLY where value > 0
        # Since your code logs every 10 epochs, we calculate the epoch as i * 10
        valid_points = [(i, val) for i, val in enumerate(norms) if val > 1e-6]

        if not valid_points: continue

        # Unzip into two lists: x (epochs) and y (norm values)
        epochs, vals = zip(*valid_points)

        # Plot clean lines
        ax.plot(epochs, vals, label=style['label'],
                color=style['color'], linewidth=2.5,
                marker=style['marker'], markersize=8, linestyle=style['ls'])

    # 3. Formatting
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Perturbation Norm ||ε||₂', fontsize=12)
    ax.set_title('Perturbation Norm Stability', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11, loc='right')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 0.06)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / save_path, dpi=300)
    print(f"✅ Saved clean plot: {save_path}")
    plt.show()
def plot_generalization_gap(stats_data, save_path='fig3_gen_gap_all.png'):
    fig, ax = plt.subplots(figsize=(10, 6))
    for method, style in METHODS.items():
        if method not in stats_data: continue
        # Calculate gap from means
        gap = stats_data[method]['train_acc']['mean'] - stats_data[method]['test_acc']['mean']
        epochs = np.arange(len(gap)) + 1
        ax.plot(epochs, gap, label=style['label'], color=style['color'], ls=style['ls'], lw=2)

    ax.set_title('Generalization Gap (Lower is Better)', fontweight='bold')
    ax.set_ylabel('Gap (%)'); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(RESULTS_DIR / save_path, dpi=300); plt.show()

# =========================================================================
# MAIN EXECUTION
# =========================================================================
if __name__ == "__main__":
    raw_data, stats_data = load_all_runs()

    if stats_data:
        print("Generating plots...")
        plot_test_accuracy(stats_data)
        plot_loss_curves(stats_data)  # <--- NEW FUNCTION
        plot_perturbation_norms(stats_data)
        plot_generalization_gap(stats_data)
    else:
        print("No data found!")